# 05. Embeddings contextuales con BERT y ELMo

En este notebook generaresmos los embeddings contextuales para utlizarlos más tarde.


## 1. Imports y configuración



In [1]:
import os
from pathlib import Path

import numpy as np
import pandas as pd
from tqdm import tqdm

from sklearn.model_selection import train_test_split

# Control de ejecución
RUN_BERT = True
RUN_ELMO = True

# Poolings que vamos a generar
POOLINGS = ["mean", "max"]

# Parámetros generales
RANDOM_STATE = 42
TEST_SIZE = 0.2

# Para BERT
BERT_MODEL_NAME = "bert-base-uncased"
BERT_MAX_LENGTH = 256
BERT_BATCH_SIZE = 8

# Para ELMo
ELMO_BATCH_SIZE = 4

# Directorio de salida
OUT_DIR = Path("/content/PLN_data") if Path("/content").exists() else Path("../Data/processed")
OUT_DIR.mkdir(parents=True, exist_ok=True)

print("Directorio de salida:", OUT_DIR.resolve())

Directorio de salida: C:\content\PLN_data


## 2. Carga del dataset

In [2]:
# Intentamos localizar el dataset de forma flexible.
possible_paths = [
    Path("/content/tcga_simple_train.csv"),      # Colab, subido al entorno
    Path("tcga_simple_train.csv"),               # mismo directorio que el notebook/script
    Path("../Data/raw/tcga_simple_train.csv"),   # estructura local del proyecto
    Path("../Data/tcga_simple_train.csv"),
]

DATA_PATH = None
for path in possible_paths:
    if path.exists():
        DATA_PATH = path
        break

if DATA_PATH is None:
    raise FileNotFoundError(
        "No se ha encontrado tcga_simple_train.csv. "
        "Sube el archivo a Colab o ajusta manualmente DATA_PATH."
    )

print("Dataset encontrado en:", DATA_PATH)

df = pd.read_csv(DATA_PATH)
df = df.dropna(subset=["text", "t"]).copy()

df["text"] = df["text"].astype(str)
df["t"] = df["t"].astype(str).str.upper().str.strip()

print(df.shape)
df.head()

Dataset encontrado en: ..\Data\raw\tcga_simple_train.csv
(5158, 3)


,patient_id,text,t
0,TCGA-BP-5195,Date of Recelpt: Clinical Diagnosis & History:...,T1
1,TCGA-D7-8573,"Material: 1) Material: stomach, Method of coll...",T3
2,TCGA-EI-7004,page 1 / 1. copy No. 3. Examination: Histopath...,T4
3,TCGA-EB-A82B,Patient ID: Gross Description: A mass is locat...,T4
4,TCGA-A6-3808,SPECIMEN. Right colon. CLINICAL NOTES. PRE-OP ...,T3


In [3]:
X_train, X_test, y_train, y_test = train_test_split(
    df["text"],
    df["t"],
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=df["t"]
)

print("Train:", X_train.shape)
print("Test:", X_test.shape)
print("Distribución train:")
print(y_train.value_counts())
print("\nDistribución test:")
print(y_test.value_counts())

# Guardamos también las etiquetas para reutilizarlas junto con los embeddings.
np.save(OUT_DIR / "y_train.npy", y_train.values)
np.save(OUT_DIR / "y_test.npy", y_test.values)

Train: (4126,)
Test: (1032,)
Distribución train:
t
T2    1401
T3    1266
T1    1042
T4     417
Name: count, dtype: int64

Distribución test:
t
T2    351
T3    317
T1    260
T4    104
Name: count, dtype: int64


## 3. Embeddings contextuales con BERT

En esta parte usamos `bert-base-uncased` para obtener representaciones contextuales. BERT devuelve un vector por token, por lo que aplicamos `mean pooling` o `max pooling` para obtener un único vector por informe.

En el pooling se tiene en cuenta la máscara de atención para no mezclar el padding con el contenido real del texto.

In [4]:
if RUN_BERT:
    import torch
    from transformers import AutoTokenizer, AutoModel

    bert_device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print("Dispositivo BERT:", bert_device)

    tokenizer = AutoTokenizer.from_pretrained(BERT_MODEL_NAME)
    bert_model = AutoModel.from_pretrained(BERT_MODEL_NAME)

    bert_model.eval()
    bert_model.to(bert_device)
else:
    print("Bloque BERT desactivado.")

Dispositivo BERT: cuda


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [5]:
def bert_embedding(texts, pooling="mean", batch_size=BERT_BATCH_SIZE, max_length=BERT_MAX_LENGTH):
    '''
    Genera embeddings BERT para una lista o Serie de textos.

    pooling = "mean": media de los vectores de tokens válidos.
    pooling = "max": máximo de los vectores de tokens válidos.
    '''
    if pooling not in ["mean", "max"]:
        raise ValueError("pooling debe ser 'mean' o 'max'")

    texts = texts.astype(str).tolist() if hasattr(texts, "astype") else list(texts)
    all_vectors = []

    for i in tqdm(range(0, len(texts), batch_size), desc=f"BERT {pooling}"):
        batch_texts = texts[i:i + batch_size]

        encoded = tokenizer(
            batch_texts,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=max_length
        )

        encoded = {k: v.to(bert_device) for k, v in encoded.items()}

        with torch.no_grad():
            outputs = bert_model(**encoded)

        # last_hidden_state: (batch, seq_len, hidden_size)
        hidden = outputs.last_hidden_state
        attention_mask = encoded["attention_mask"].bool()

        # Quitamos tokens especiales de forma sencilla:
        # [CLS] suele estar en la primera posición y [SEP] en la última posición válida.
        valid_mask = attention_mask.clone()
        valid_mask[:, 0] = False

        lengths = attention_mask.sum(dim=1)
        for row, length in enumerate(lengths):
            sep_index = int(length.item()) - 1
            if sep_index >= 0:
                valid_mask[row, sep_index] = False

        # Evitamos divisiones entre cero si algún texto quedara vacío tras quitar especiales.
        counts = valid_mask.sum(dim=1).clamp(min=1)

        if pooling == "mean":
            masked_hidden = hidden * valid_mask.unsqueeze(-1)
            batch_vectors = masked_hidden.sum(dim=1) / counts.unsqueeze(-1)

        else:
            masked_hidden = hidden.masked_fill(~valid_mask.unsqueeze(-1), -1e9)
            batch_vectors = masked_hidden.max(dim=1).values

        all_vectors.append(batch_vectors.cpu().numpy())

    return np.vstack(all_vectors)

In [6]:
if RUN_BERT:
    for pooling in POOLINGS:
        print(f"Generando BERT {pooling}...")

        X_train_bert = bert_embedding(X_train, pooling=pooling)
        X_test_bert = bert_embedding(X_test, pooling=pooling)

        np.save(OUT_DIR / f"X_train_bert_{pooling}.npy", X_train_bert)
        np.save(OUT_DIR / f"X_test_bert_{pooling}.npy", X_test_bert)

        print(f"BERT {pooling} train:", X_train_bert.shape)
        print(f"BERT {pooling} test:", X_test_bert.shape)
        print("-" * 60)

Generando BERT mean...


BERT mean: 100%|██████████| 129/129 [00:12<00:00, 10.23it/s]


BERT mean train: (4126, 768)
BERT mean test: (1032, 768)
------------------------------------------------------------
Generando BERT max...


BERT max: 100%|██████████| 129/129 [00:13<00:00,  9.75it/s]

BERT max train: (4126, 768)
BERT max test: (1032, 768)
------------------------------------------------------------


## 4. Embeddings contextuales con ELMo

Ahora hacemos lo mismo con ELMo. Este modelo también devuelve una secuencia de vectores contextualizados, en este caso de 1024 dimensiones. De nuevo aplicamos `mean pooling` y `max pooling` para obtener una representación fija por informe.

In [7]:
if RUN_ELMO:
    import tensorflow as tf
    import tensorflow_hub as hub

    print("TensorFlow:", tf.__version__)
    print("GPUs disponibles:", tf.config.list_physical_devices("GPU"))

    # Cargamos ELMo desde TensorFlow Hub.
    # trainable=False porque no queremos ajustar ELMo, solo extraer características.
    with tf.device("/GPU:0" if tf.config.list_physical_devices("GPU") else "/CPU:0"):
        elmo = hub.KerasLayer(
            "https://tfhub.dev/google/elmo/3",
            signature="default",
            output_key="elmo",
            trainable=False,
            dtype=tf.string
        )

    print("ELMo cargado correctamente.")
else:
    print("Bloque ELMo desactivado.")

ModuleNotFoundError: No module named 'pkg_resources'

In [ ]:
def elmo_embedding(texts, pooling="mean", batch_size=ELMO_BATCH_SIZE):
    '''
    Genera embeddings ELMo para una lista o Serie de textos.

    pooling = "mean": media sobre la dimensión temporal.
    pooling = "max": máximo sobre la dimensión temporal.
    '''
    if pooling not in ["mean", "max"]:
        raise ValueError("pooling debe ser 'mean' o 'max'")

    texts = texts.astype(str).tolist() if hasattr(texts, "astype") else list(texts)
    all_vectors = []

    for i in tqdm(range(0, len(texts), batch_size), desc=f"ELMo {pooling}"):
        batch_texts = texts[i:i + batch_size]

        # batch_embeddings: (batch_size, seq_len, 1024)
        batch_embeddings = elmo(tf.constant(batch_texts)).numpy()

        if pooling == "mean":
            batch_vectors = np.mean(batch_embeddings, axis=1)
        else:
            batch_vectors = np.max(batch_embeddings, axis=1)

        all_vectors.append(batch_vectors)

    return np.vstack(all_vectors)

In [ ]:
if RUN_ELMO:
    for pooling in POOLINGS:
        print(f"Generando ELMo {pooling}...")

        X_train_elmo = elmo_embedding(X_train, pooling=pooling)
        X_test_elmo = elmo_embedding(X_test, pooling=pooling)

        np.save(OUT_DIR / f"X_train_elmo_{pooling}.npy", X_train_elmo)
        np.save(OUT_DIR / f"X_test_elmo_{pooling}.npy", X_test_elmo)

        print(f"ELMo {pooling} train:", X_train_elmo.shape)
        print(f"ELMo {pooling} test:", X_test_elmo.shape)
        print("-" * 60)

## 5. Comprobación de archivos generados

Por último, listamos los archivos `.npy` creados. Estos vectores se usarán después como entrada para modelos clásicos o redes densas sin tener que recalcular BERT o ELMo cada vez.

In [ ]:
print("Archivos guardados en:", OUT_DIR.resolve())

for file in sorted(OUT_DIR.glob("*.npy")):
    print(file.name)